# ACB Experiments — Free Replication on Google Colab

This notebook runs the three experiments (P1, P2, P3) from the paper
*The Agent Coordination Bound (ACB)* using **free GPU on Colab + Ollama**.

**Requirements:** Colab with GPU runtime (Runtime → Change runtime type → T4 GPU).

**Estimated runtime:**
- Quick test mode: ~20-40 min
- Full replication (50 reps × all tasks): ~24-48h

---
## 1. Install Ollama

In [ ]:
# Step 1a: Install system dependencies (zstd is required by Ollama)
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd pciutils > /dev/null 2>&1
print('✓ System dependencies installed')

In [ ]:
# Step 1b: Install Ollama
!curl -fsSL https://ollama.ai/install.sh | sh
print('\n✓ Ollama installed')

In [ ]:
# Step 1c: Start Ollama server in background
import subprocess, time, os

os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'

proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=open('/tmp/ollama_stdout.log', 'w'),
    stderr=open('/tmp/ollama_stderr.log', 'w'),
)
print(f'Ollama server started (PID: {proc.pid})')
print('Waiting for server to be ready...')
time.sleep(10)

# Verify it's running
import urllib.request
try:
    resp = urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=5)
    print('✓ Ollama server is running')
except Exception as e:
    print(f'✗ Server not responding: {e}')
    print('Stderr log:')
    !cat /tmp/ollama_stderr.log | tail -20

In [ ]:
# Step 1d: Pull a model
# For T4 free tier (16GB VRAM): use llama3.1:8b or mistral:7b
# For L4/A100 (Colab Pro): use llama3.1:70b for best results

MODEL = 'llama3.1:8b'  # Change this if you have more VRAM

print(f'Pulling {MODEL}... (this may take 5-10 minutes)')
!ollama pull {MODEL}
print(f'\n✓ {MODEL} ready')

In [ ]:
# Step 1e: Quick test — make sure the model responds
!ollama run {MODEL} 'What is 2+2? Answer with just the number.' --verbose 2>&1 | head -5
print('\n✓ Model is working')

---
## 2. Install ACB

In [ ]:
# Clone the repository
!git clone https://github.com/<YOUR-USERNAME>/acb-experiments.git 2>/dev/null || echo 'Already cloned'
%cd acb-experiments

# Install Python dependencies
!pip install numpy scipy pandas matplotlib seaborn pyyaml httpx python-dotenv tqdm -q
print('\n✓ Dependencies installed')

In [ ]:
# Configure LLM backend to use local Ollama
import os
os.environ['LOCAL_MODEL_URL'] = 'http://127.0.0.1:11434'
os.environ['LOCAL_MODEL_NAME'] = MODEL
os.environ['MAX_CONCURRENT'] = '1'
os.environ['OUTPUT_DIR'] = 'results/'
os.environ['SEED'] = '42'

with open('.env', 'w') as f:
    f.write(f'LOCAL_MODEL_URL=http://127.0.0.1:11434\n')
    f.write(f'LOCAL_MODEL_NAME={MODEL}\n')
    f.write(f'MAX_CONCURRENT=1\n')
    f.write(f'SEED=42\n')

print('✓ Environment configured')
print(f'  Model: {MODEL}')
print(f'  Backend: Ollama @ http://127.0.0.1:11434')

In [ ]:
# Download benchmark datasets
!PYTHONPATH=. python benchmarks/setup.py

---
## 3. Validate the Math (no GPU needed)

These tests confirm the analytical formulas are correctly implemented.
They run in seconds and don't call any LLM.

In [ ]:
import sys
sys.path.insert(0, '.')

# Monte Carlo validation of P(harm|n) — reproduces Table 3
from monte_carlo.validate_pharm import run_validation
results = run_validation(mc_runs=50000)

In [ ]:
# CBI diagnostic — reproduces Table 6
from acb.cbi import interpret_cbi

configs = [
    ('AutoGen GroupChat (3 agents)', 3, 0.72, 0.082),
    ('LangChain 5-agent workflow',   5, 0.72, 0.082),
    ('AgentPrune BEFORE pruning',   20, 0.51, 0.065),
    ('AgentPrune AFTER pruning',     8, 0.51, 0.065),
    ('Self-consistency k=40',       40, 0.56, 0.041),
]

for name, n, a, c in configs:
    r = interpret_cbi(n, a, c)
    print(f'{name}:')
    print(f'  {r}\n')

In [ ]:
# Greedy fleet selection — reproduces Table 7
from acb.greedy_fleet import greedy_fleet_select

agents = [
    ('Claude-3.5-Sonnet', 0.89),
    ('GPT-4o', 0.87),
    ('GPT-4o-inst2', 0.86),
    ('GPT-4-Turbo', 0.82),
    ('Gemini-1.5-Pro', 0.79),
    ('GPT-4o-mini', 0.72),
    ('LLaMA-3-70B', 0.57),
]

result = greedy_fleet_select(agents, c=0.082, cross_model_penalty=1.3)

print('Greedy Heterogeneous Fleet Selection')
print('=' * 65)
for step in result.steps:
    stop = ' ← STOP' if step.stop else ''
    print(f'  Step {step.step}: {step.agent_name:25s} ΔI={step.marginal_gain:+.4f}  I={step.cumulative_I:.4f}{stop}')
print(f'\nSelected: {len(result.selected)} agents → I(S*) = {result.total_I:.4f}')

In [ ]:
# Generate paper figures
from analysis.plots import generate_all_figures
generate_all_figures('figures/')

from IPython.display import Image, display
for fig in ['fig1a_performance_curve.png', 'fig3a_pharm_validation.png', 'fig2b_cbi_dashboard.png']:
    print(f'\n--- {fig} ---')
    display(Image(f'figures/{fig}', width=600))

---
## 4. Run Live Experiments

**Start with quick mode** (2 reps, 10 tasks, 3 fleet sizes) to make sure
the LLM backend works. Then switch to full mode.

⚠️ **Full replication takes many hours on Colab free tier.** Consider:
- Running one experiment at a time
- Saving results to Google Drive between experiments
- Using Colab Pro for longer runtime limits

In [ ]:
# QUICK TEST (~20-40 min) — run this first
!PYTHONPATH=. python run_all.py --quick --output-dir results/quick/

In [ ]:
# Check quick test results
import json, glob

for result_file in sorted(glob.glob('results/quick/*.json')):
    with open(result_file) as f:
        data = json.load(f)
    print(f'\n{"=" * 60}')
    print(f'Experiment: {data.get("experiment_name", "unknown")}')
    print(f'{"=" * 60}')
    summary = data.get('summary', {})
    for k, v in summary.items():
        if k != 'llm_usage':
            print(f'  {k}: {v}')
    usage = summary.get('llm_usage', {})
    if usage:
        print(f'  Total LLM calls: {usage.get("total_calls", 0)}')
        print(f'  Total tokens: {usage.get("total_tokens", 0):,}')

### 4b. Full Experiments (run individually)

Once the quick test passes, run each experiment separately.
Save to Google Drive after each one in case the runtime disconnects.

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/acb-results

In [ ]:
# P1: All-to-all fleet sizing on HumanEval
!PYTHONPATH=. python -m experiments.p1_fleet_sizing \
    --fleet-sizes 1 2 3 4 5 6 7 8 9 10 12 15 \
    --reps 50 \
    --output-dir results/p1/

!cp -r results/p1/ /content/drive/MyDrive/acb-results/p1/
print('\n✓ P1 results saved to Google Drive')

In [ ]:
# P2: Supervisor vs all-to-all on MATH
!PYTHONPATH=. python -m experiments.p2_topology_crossover \
    --fleet-sizes 1 2 3 4 5 6 7 8 9 10 12 15 \
    --reps 50 \
    --max-tasks 200 \
    --output-dir results/p2/

!cp -r results/p2/ /content/drive/MyDrive/acb-results/p2/
print('\n✓ P2 results saved to Google Drive')

In [ ]:
# P3: Shared vs isolated RAG context on MATH
!PYTHONPATH=. python -m experiments.p3_rag_diversity \
    --fleet-sizes 1 3 6 9 \
    --reps 50 \
    --max-tasks 200 \
    --output-dir results/p3/

!cp -r results/p3/ /content/drive/MyDrive/acb-results/p3/
print('\n✓ P3 results saved to Google Drive')

---
## 5. Analyze Results

In [ ]:
import json, glob

result_files = sorted(
    glob.glob('results/**/*.json', recursive=True) +
    glob.glob('/content/drive/MyDrive/acb-results/**/*.json', recursive=True)
)

for result_file in result_files:
    with open(result_file) as f:
        data = json.load(f)
    print(f'\n{"=" * 60}')
    print(f'Experiment: {data.get("experiment_name", "unknown")}')
    print(f'{"=" * 60}')
    summary = data.get('summary', {})
    for k, v in summary.items():
        if k != 'llm_usage':
            print(f'  {k}: {v}')

In [ ]:
# P1 key result: does the empirical peak match n*?
p1_files = glob.glob('results/p1/*.json') + glob.glob('/content/drive/MyDrive/acb-results/p1/*.json')
if p1_files:
    with open(p1_files[-1]) as f:
        p1 = json.load(f)
    s = p1['summary']
    print('P1 RESULT: Performance Peak Prediction')
    print(f'  a (Pass@1, n=1) = {s["a"]:.4f}')
    print(f'  c (overhead)    = {s["c"]:.4f}')
    print(f'  n* (predicted)  = {s["n_star"]}')
    print(f'  Empirical peak  = {s["empirical_peak"]}')
    match = '✓ CONFIRMED' if s['n_star_matches'] else '✗ FALSIFIED'
    print(f'  Result:           {match}')
    print(f'\n  Pass@1 by fleet size:')
    for n, acc in sorted(s['pass_at_1'].items(), key=lambda x: int(x[0])):
        bar = '█' * int(acc * 40)
        print(f'    n={int(n):2d}: {acc:.3f} {bar}')
else:
    print('No P1 results found. Run experiment first.')

In [ ]:
# P2 key result: does supervisor beat all-to-all at every n >= 2?
p2_files = glob.glob('results/p2/*.json') + glob.glob('/content/drive/MyDrive/acb-results/p2/*.json')
if p2_files:
    with open(p2_files[-1]) as f:
        p2 = json.load(f)
    s = p2['summary']
    print('P2 RESULT: Topology Crossover')
    print(f'  Prediction confirmed: {s.get("p2_confirmed", "N/A")}')
    print(f'  Prediction falsified: {s.get("p2_falsified", "N/A")}')
    print(f'\n  Accuracy by fleet size and topology:')
    for n, stats in sorted(s.get('comparisons', {}).items(), key=lambda x: int(x[0])):
        winner = 'SUP ✓' if stats['sup_wins'] else 'A2A'
        sig = ' *' if stats.get('significant', False) else ''
        print(f'    n={int(n):2d}: a2a={stats["a2a_acc"]:.3f}  sup={stats["sup_acc"]:.3f}  → {winner}{sig}')
else:
    print('No P2 results found. Run experiment first.')

In [ ]:
# P3 key result: does shared context kill diversity?
p3_files = glob.glob('results/p3/*.json') + glob.glob('/content/drive/MyDrive/acb-results/p3/*.json')
if p3_files:
    with open(p3_files[-1]) as f:
        p3 = json.load(f)
    s = p3['summary']
    print('P3 RESULT: Context Diversity')
    print(f'  Baseline (n=1) accuracy: {s.get("baseline_acc", "N/A")}')
    print(f'  rho_crit = {s.get("rho_crit", "N/A")}')
    print(f'  Shared context no benefit: {s.get("p3_shared_no_benefit", "N/A")}')
    print(f'  Isolated context benefit:  {s.get("p3_isolated_benefit", "N/A")}')
    print(f'  P3 confirmed: {s.get("p3_confirmed", "N/A")}')
    print(f'\n  By fleet size:')
    for n, stats in sorted(s.get('by_fleet_size', {}).items(), key=lambda x: int(x[0])):
        rho_sh = stats.get('rho_shared_mean', None)
        rho_iso = stats.get('rho_isolated_mean', None)
        rho_str = f'rho_sh={rho_sh:.3f} rho_iso={rho_iso:.3f}' if rho_sh is not None else ''
        print(f'    n={int(n):2d}: shared={stats["shared_acc"]:.3f}  isolated={stats["isolated_acc"]:.3f}  {rho_str}')
else:
    print('No P3 results found. Run experiment first.')